# Assignment 11.1 — Demand Prediction and Pricing Optimization with Gurobi

Building a **demand prediction model** and a **pricing optimization model** for an amusement park souvenir, using:

- A **linear additive demand model** with price, lagged prices, and seasonality  
- A **discrete price ladder** as a business rule  
- A **revenue‑maximizing objective** over Weeks 157–164  

We need to extract and plot the **optimal weekly prices**.

In [ ]:
# Install and import Gurobi and Plotly packages

!pip install gurobipy plotly --quiet

In [1]:
# Import the packages 
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import plotly.express as px

## 1. Planning Horizon and Indexing

We consider Weeks 157–164 as the planning horizon.  
We will model all eight weeks, but per the assignment, Week 164 does not need to appear in the final output table of predicted prices.

In [2]:
# Define planning horizon: weeks 157–164
weeks = list(range(157, 165))
print("Weeks (t):", weeks)

# Convenience aliases for the first two weeks
w1 = weeks[0]
w2 = weeks[1]
w1, w2

Weeks (t): [157, 158, 159, 160, 161, 162, 163, 164]


(157, 158)

## 2. Demand Model Parameters

The demand model is a **linear additive regression**:

$$
d_t = 1.978242858 
- 2.809634145\,p_t 
+ 0.963410728\,p_{t-1} 
+ 0.759639170\,p_{t-2}
+ \sum_{s=2}^{13} \beta_s \cdot \text{Season}_s
$$

This captures:

- Own‑price effect (negative coefficient on $(p_t)$)  
- Lagged price effects ($(p_{t-1}, p_{t-2})$)  
- Seasonal variation via dummy variables $(\text{Season}_2, \dots, \text{Season}_{13})$.

In [3]:
# Demand model parameters (from assignment)
intercept = 1.978242858

p_coeff  = -2.809634145   # coefficient on p_t
p1_coeff =  0.963410728   # coefficient on p_{t-1}
p2_coeff =  0.759639170   # coefficient on p_{t-2}

# Seasonal coefficients: Season1 is baseline (0), Seasons 2–13 have explicit coefficients
season_coeff = {
    1: 0.0,
    2: -0.562046910,
    3:  0.087545274,
    4: -0.402637480,
    5: -0.027326010,
    6:  0.004349885,
    7: -0.036102297,
    8: -0.069280527,
    9:  0.160276197,
    10: 1.104208897,
    11: 1.122711287,
    12: 1.176802194,
    13: 0.947945548
}

## 3. Season Mapping

The year is divided into **13 seasons**, each consisting of 4 weeks.  
We map each week in the planning horizon to a season index $(s \in \{1,\dots,13\})$.

In [4]:
# Map each week to a season (1–13), assuming 52 weeks/year and 4 weeks/season
season = {}
for w in weeks:
    week_in_year = ((w - 1) % 52) + 1  # wrap into 1–52
    s = int(np.ceil(week_in_year / 4)) # 4 weeks per season
    season[w] = s

season

{157: 1, 158: 1, 159: 1, 160: 1, 161: 2, 162: 2, 163: 2, 164: 2}

## 4. Price Ladder (Business Rule)

Prices must be chosen from the following discrete **price ladder**:

- 1.00  
- 0.95  
- 0.85  
- 0.75  
- 0.60  
- 0.50  

We will enforce this using **binary decision variables** that select exactly one ladder price per week.

In [5]:
# Six-level price ladder
price_ladder = [1.00, 0.95, 0.85, 0.75, 0.60, 0.50]
price_ladder

[1.0, 0.95, 0.85, 0.75, 0.6, 0.5]

## 5. Optimization Model with Gurobi

We now construct a **mixed‑integer quadratic program**:

- **Decision variables**
  - Continuous: $(p_t)$ (price in week t)
  - Binary: $(x_{t,k})$ (1 if ladder price k is chosen in week t, 0 otherwise)

- **Objective function**
  - Maximize total revenue: $(\sum_t p_t \cdot d_t)$

- **Constraints**
  - Price ladder selection: exactly one ladder price per week
  - Price definition: $(p_t = \sum_k k \cdot x_{t,k})$
  - Demand defined by the linear additive model

In [6]:
# Create Gurobi model
mod = gp.Model("amusement_souvenir_pricing")

# Decision variables
p = mod.addVars(weeks, name="price")                          # continuous prices
x = mod.addVars(weeks, price_ladder, vtype=GRB.BINARY, name="x")  # ladder selection

Restricted license - for non-production use only - expires 2027-11-29
